
# 混合戦略の期待利得計算ツール

このノートブックでは，プレイヤー数と各プレイヤーの戦略数を入力して，有限戦略型ゲームを作成する．  
各戦略プロフィールに対する利得を手で入力し，各プレイヤーの混合戦略の確率を入力すると，各プレイヤーの期待利得を計算できる．

## 使い方
1. プレイヤー数を入力する．
2. 各プレイヤーの戦略数をカンマ区切りで入力する．
3. 「ゲーム表を作成」を押す．
4. 各戦略プロフィールに対する利得を入力する．
5. 各プレイヤーの混合戦略の確率を入力する．
6. 「期待利得を計算」を押す．


In [1]:

import itertools
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

state = {
    "n_players": None,
    "strategy_counts": None,
    "strategy_names": None,
    "profiles": None,
    "payoff_widgets": None,
    "prob_widgets": None,
}

TOL = 1e-8

def parse_strategy_counts(text, n_players):
    counts = [int(x.strip()) for x in text.split(",") if x.strip() != ""]
    if len(counts) != n_players:
        raise ValueError("プレイヤー数と戦略数の個数が一致していません．")
    if any(c <= 0 for c in counts):
        raise ValueError("戦略数はすべて正の整数にしてください．")
    return counts

def make_strategy_names(strategy_counts):
    strategy_names = []
    for i, c in enumerate(strategy_counts, start=1):
        names = [f"P{i}_S{j}" for j in range(1, c + 1)]
        strategy_names.append(names)
    return strategy_names

def profile_label(profile):
    return "(" + ", ".join(profile) + ")"

def get_payoff_array():
    profiles = state["profiles"]
    n_players = state["n_players"]
    payoff_widgets = state["payoff_widgets"]
    payoffs = {}
    for profile in profiles:
        payoffs[profile] = []
        for i in range(n_players):
            payoffs[profile].append(payoff_widgets[(profile, i)].value)
    return payoffs

def get_probability_vectors():
    prob_widgets = state["prob_widgets"]
    strategy_names = state["strategy_names"]
    probs = []
    for i, names in enumerate(strategy_names):
        p = np.array([prob_widgets[(i, s)].value for s in names], dtype=float)
        probs.append(p)
    return probs

def check_probabilities(probs):
    messages = []
    ok = True
    for i, p in enumerate(probs, start=1):
        if np.any(p < -TOL):
            ok = False
            messages.append(f"プレイヤー{i}：負の確率が含まれています．")
        if abs(np.sum(p) - 1.0) > 1e-6:
            ok = False
            messages.append(f"プレイヤー{i}：確率の合計が1ではありません．現在の合計は {np.sum(p):.6f} です．")
    return ok, messages

def compute_expected_payoffs():
    n_players = state["n_players"]
    strategy_names = state["strategy_names"]
    profiles = state["profiles"]
    payoffs = get_payoff_array()
    probs = get_probability_vectors()
    ok, messages = check_probabilities(probs)
    if not ok:
        return None, messages
    expected = np.zeros(n_players)
    for profile in profiles:
        prob = 1.0
        for i, strategy in enumerate(profile):
            idx = strategy_names[i].index(strategy)
            prob *= probs[i][idx]
        for i in range(n_players):
            expected[i] += prob * payoffs[profile][i]
    return expected, []

def create_payoff_input_table():
    n_players = state["n_players"]
    profiles = state["profiles"]
    header = [widgets.HTML("<b>戦略プロフィール</b>", layout=widgets.Layout(width="220px"))]
    for i in range(n_players):
        header.append(widgets.HTML(f"<b>P{i+1} の利得</b>", layout=widgets.Layout(width="130px")))
    rows = [widgets.HBox(header)]
    payoff_widgets = {}
    for profile in profiles:
        row = [widgets.HTML(profile_label(profile), layout=widgets.Layout(width="220px"))]
        for i in range(n_players):
            w = widgets.FloatText(value=0.0, layout=widgets.Layout(width="130px"))
            payoff_widgets[(profile, i)] = w
            row.append(w)
        rows.append(widgets.HBox(row))
    state["payoff_widgets"] = payoff_widgets
    return widgets.VBox(rows)

def create_probability_inputs():
    strategy_names = state["strategy_names"]
    prob_widgets = {}
    blocks = []
    for i, names in enumerate(strategy_names):
        title = widgets.HTML(f"<b>プレイヤー{i+1} の混合戦略</b>")
        row = []
        default = 1.0 / len(names)
        for s in names:
            w = widgets.FloatText(
                value=default,
                description=s,
                layout=widgets.Layout(width="160px"),
                style={"description_width": "60px"}
            )
            prob_widgets[(i, s)] = w
            row.append(w)
        blocks.append(widgets.VBox([title, widgets.HBox(row)]))
    state["prob_widgets"] = prob_widgets
    return widgets.VBox(blocks)

def payoff_matrix_for_two_players():
    if state["n_players"] != 2:
        return None
    strategy_names = state["strategy_names"]
    payoffs = get_payoff_array()
    rows = strategy_names[0]
    cols = strategy_names[1]
    data = []
    for s1 in rows:
        row = []
        for s2 in cols:
            profile = (s1, s2)
            u = payoffs[profile]
            row.append(f"({u[0]:.3g}, {u[1]:.3g})")
        data.append(row)
    return pd.DataFrame(data, index=rows, columns=cols)

def payoff_table_for_n_players():
    profiles = state["profiles"]
    n_players = state["n_players"]
    payoffs = get_payoff_array()
    records = []
    for profile in profiles:
        record = {}
        for i, s in enumerate(profile, start=1):
            record[f"P{i} 戦略"] = s
        for i in range(n_players):
            record[f"P{i+1} 利得"] = payoffs[profile][i]
        records.append(record)
    return pd.DataFrame(records)

def normalize_probabilities(_=None):
    prob_widgets = state.get("prob_widgets")
    strategy_names = state.get("strategy_names")
    if prob_widgets is None:
        return
    for i, names in enumerate(strategy_names):
        vals = np.array([prob_widgets[(i, s)].value for s in names], dtype=float)
        vals = np.maximum(vals, 0)
        total = vals.sum()
        if total <= TOL:
            vals[:] = 1.0 / len(vals)
        else:
            vals = vals / total
        for s, v in zip(names, vals):
            prob_widgets[(i, s)].value = float(v)

n_players_widget = widgets.BoundedIntText(
    value=2, min=1, max=6, description="プレイヤー数",
    layout=widgets.Layout(width="220px")
)
strategy_counts_widget = widgets.Text(
    value="2,2", description="戦略数",
    placeholder="例：2,2 または 2,3,2",
    layout=widgets.Layout(width="420px")
)
create_button = widgets.Button(description="ゲーム表を作成", button_style="primary")
normalize_button = widgets.Button(description="確率を正規化")
calc_button = widgets.Button(description="期待利得を計算", button_style="success")
show_matrix_button = widgets.Button(description="利得表を表示")
game_output = widgets.Output()
result_output = widgets.Output()

def on_create_clicked(_):
    with game_output:
        clear_output()
        try:
            n_players = n_players_widget.value
            strategy_counts = parse_strategy_counts(strategy_counts_widget.value, n_players)
            strategy_names = make_strategy_names(strategy_counts)
            profiles = list(itertools.product(*strategy_names))
            state["n_players"] = n_players
            state["strategy_counts"] = strategy_counts
            state["strategy_names"] = strategy_names
            state["profiles"] = profiles
            print("ゲーム表を作成しました．")
            print("プレイヤー数：", n_players)
            print("戦略数：", strategy_counts)
            print("戦略プロフィール数：", len(profiles))
            print()
            if len(profiles) > 200:
                print("注意：戦略プロフィール数が多いため，入力欄が非常に多くなります．")
            display(widgets.HTML("<h3>利得の入力</h3>"))
            display(create_payoff_input_table())
            display(widgets.HTML("<h3>混合戦略の確率入力</h3>"))
            display(create_probability_inputs())
            display(widgets.HBox([normalize_button, show_matrix_button, calc_button]))
        except Exception as e:
            print("エラー：", e)

def on_show_matrix_clicked(_):
    with result_output:
        clear_output()
        if state["profiles"] is None:
            print("先にゲーム表を作成してください．")
            return
        if state["n_players"] == 2:
            print("2人ゲームの利得行列")
            display(payoff_matrix_for_two_players())
        else:
            print("3人以上のゲームなので，戦略プロフィール一覧で表示します．")
            display(payoff_table_for_n_players())

def on_calc_clicked(_):
    with result_output:
        clear_output()
        if state["profiles"] is None:
            print("先にゲーム表を作成してください．")
            return
        expected, messages = compute_expected_payoffs()
        if messages:
            print("確率の入力に問題があります．")
            for m in messages:
                print("-", m)
            return
        df = pd.DataFrame(
            [{"プレイヤー": f"P{i+1}", "期待利得": expected[i]} for i in range(len(expected))]
        )
        print("期待利得")
        display(df)
        print("現在の利得表")
        if state["n_players"] == 2:
            display(payoff_matrix_for_two_players())
        else:
            display(payoff_table_for_n_players())

create_button.on_click(on_create_clicked)
normalize_button.on_click(normalize_probabilities)
show_matrix_button.on_click(on_show_matrix_clicked)
calc_button.on_click(on_calc_clicked)

display(widgets.VBox([
    widgets.HTML("<h2>ゲームの設定</h2>"),
    widgets.HBox([n_players_widget, strategy_counts_widget, create_button]),
    game_output,
    result_output
]))



## 例：2人2戦略ゲーム

プレイヤー数を `2`，戦略数を `2,2` としてゲーム表を作成する．

例えば，次の利得行列を入力する．

| 1＼2 | L        | R        |
|------|----------|----------|
| U    | $(1,-1)$ | $(-1,1)$ |
| D    | $(-1,1)$ | $(1,-1)$ |

このツールでは，戦略名は自動的に

- プレイヤー1：`P1_S1`, `P1_S2`
- プレイヤー2：`P2_S1`, `P2_S2`

として表示される．  
したがって，`P1_S1` を $U$，`P1_S2` を $D$，`P2_S1` を $L$，`P2_S2` を $R$ と読み替えればよい．
